In [1]:
import os

In [2]:
%pwd

'c:\\Users\\dibim\\OneDrive\\Desktop\\Text_summarizer\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\dibim\\OneDrive\\Desktop\\Text_summarizer'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [6]:
from TextSummarizationProject.constants import *
from TextSummarizationProject.utils.common import read_yaml, create_directories

In [11]:
class ConfigurationManager:
    def __init__(self, 
                 config_filepath: Path = CONFIG_FILE_PATH, 
                 params_filepath: Path = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artefacts_root])
    
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        try:
            config = self.config.data_ingestion

            create_directories([config.root_dir])

            data_ingestion_config = DataIngestionConfig(
                root_dir=config.root_dir,
                source_URL=config.source_URL,
                local_data_file=config.local_data_file,
                unzip_dir=config.unzip_dir
            )

            return data_ingestion_config
        
        except Exception as e:
            raise e

### Components

In [12]:
import os
import zipfile
from TextSummarizationProject.utils.common import get_size
import urllib.request as request
from TextSummarizationProject.logging import logger

In [13]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config
    
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filname, headers = request.urlretrieve(
                url=self.config.source_URL,
                filename=self.config.local_data_file
            
            )
            logger.info(f"{filname} downloaded with following info: \n{headers}")
        else:
            logger.info(f"File already exists of size :[{get_size(Path(self.config.local_data_file))}]")

    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file to the specified directory.
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)
        

In [14]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()  
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-05-10 07:52:13,619 - INFO - common - yaml file: config\config.yaml loaded successfully]
[2026-05-10 07:52:13,623 - INFO - common - yaml file: params.yaml loaded successfully]
[2026-05-10 07:52:13,627 - INFO - common - Directory: artefacts created successfully]
[2026-05-10 07:52:13,631 - INFO - common - Directory: artefacts/data_ingestion created successfully]
[2026-05-10 07:52:18,253 - INFO - 456087986 - artefacts/data_ingestion/data.zip downloaded with following info: 
Connection: close
Content-Length: 7903594
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "dbc016a060da18070593b83afff580c9b300f0b6ea4147a7988433e04df246ca"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: B4B0:19B412:18D44B3:1C528D6:6A001D09
Accept-Ranges: bytes
Date: Sun, 10 May 2026 05:52:13 GMT
Via: 1.1 varnish
X-Ser